# BCO7006 Session 6 — Object-Oriented Programming Extensions

**Week 3 · Session 6 · 2 hours**

Last session you learned how to **build** classes — the foundations. Today you'll learn how to **extend** them:

- **Inheritance** — when you have several similar classes (Book, Electronics, Apparel) that share most of their behaviour, Python lets one class *inherit* from another so you don't repeat yourself.
- **Magic methods** — special methods like `__str__` and `__eq__` that let your objects work seamlessly with `print()`, `==`, `len()`, and Python's other built-in operations.
- **`@dataclass`** — a Python decorator that writes most of `__init__` for you. Massive boilerplate reduction.

By the end of this session, you'll be able to:

- Use inheritance to specialise a child class from a parent, including overriding methods and calling `super().__init__()`.
- Choose between inheritance and composition for a given modelling situation.
- Implement common magic methods so your classes work cleanly with Python's built-in operations.
- Use `@dataclass` to remove `__init__` and `__repr__` boilerplate from data-heavy classes.
- Recognise when `frozen=True` is appropriate — and what its limits are.

## How this notebook works

Same five-level ladder as Session 5:

| Level | What it looks like | Where |
|---|---|---|
| **L1 Mirror** | Read the instructor's code — you don't retype it | In-class |
| **L2 Trace** | Predict the output **in a markdown cell**, then run | In-class |
| **L3 Modify** | Change working code or fix a planted bug | In-class |
| **L4 Build** | Build from a spec — **Claude/Google permitted** | At-home |
| **L5 Extend** | Open-ended design task — **marked pass/fail on engagement** | At-home |

**Using Claude / AI tools on L4 and L5 is expected.** What we check is whether you understood what you got back. Every L4/L5 task ends with a short reflection — paste your prompt(s), tell us one thing the AI got right, and one thing you changed, rejected, or had to look up.

**Some cells are deliberately AI-resistant.** They ask you to compare to your own S5 code, to predict before you run, or to explain a specific `super().__init__()` call. The chatbot doesn't know your code or what's happening on your screen.

**Setup.** Open this in Google Colab. Save a copy to your drive so your work persists.

---

# Part 1: Inheritance — "is-a" relationships

In S5 you built composition: an `Order` HAS-A `Customer` and HAS-A `Cart`. Those are separate things, glued together.

Inheritance is different. Inheritance is "is-a." A `Book` **is a** `Product`. An `Electronics` item **is a** `Product`. They share most of their behaviour with Product, and they each add a little extra.

## Instructor demo — read

In [ ]:
# A generic Product — every product has an id, name, and price
class Product:
    def __init__(self, id, name, price):
        self.id = id
        self.name = name
        self.price = price

    def describe(self):
        return f"{self.name} (${self.price})"


# A Book IS A Product — and also has an author
class Book(Product):
    def __init__(self, id, name, price, author):
        super().__init__(id, name, price)   # let Product set id, name, price
        self.author = author

    # Override describe — Books describe themselves with the author
    def describe(self):
        return f"{self.name} by {self.author} (${self.price})"


# Electronics IS A Product — and also has a brand
class Electronics(Product):
    def __init__(self, id, name, price, brand):
        super().__init__(id, name, price)
        self.brand = brand

    def describe(self):
        return f"{self.brand} {self.name} (${self.price})"


book = Book("B001", "Python Crash Course", 29.99, "Eric Matthes")
laptop = Electronics("E001", "ThinkPad X1", 1499.00, "Lenovo")

print(book.describe())
print(laptop.describe())

Three things to notice:

1. **`class Book(Product):`** — the parent class goes in parentheses. Book inherits everything Product has.
2. **`super().__init__(id, name, price)`** — this calls Product's `__init__` to set up the inherited attributes. Without this line, the Book wouldn't have `id`, `name`, or `price` at all.
3. **Overriding `describe()`** — Book defines its own `describe`. When you call `book.describe()`, Python uses Book's version, not Product's. This is called *method overriding*.

## Predict-then-run (L2)

Predict the output of these three `isinstance` calls **before** running the next cell.

**Your predictions:**

- `isinstance(book, Book)` → ?
- `isinstance(book, Product)` → ?
- `isinstance(book, Electronics)` → ?

*(replace this text with your predictions)*

In [ ]:
print("isinstance(book, Book):       ", isinstance(book, Book))
print("isinstance(book, Product):    ", isinstance(book, Product))
print("isinstance(book, Electronics):", isinstance(book, Electronics))

The second one is the key insight: `book` is *both* a Book and a Product, simultaneously. Inheritance creates a family — every Book is also a Product, but not every Product is a Book.

This is why the S5 `identify_shape` example was a bit clunky. When you have a clean inheritance hierarchy, you often don't need `isinstance` chains at all — you just call the method and Python picks the right version.

## Your turn (L3a): build an Apparel class

Add an `Apparel` class that inherits from `Product`. Each Apparel item has an additional `size` attribute (e.g. `"M"`, `"L"`). Override `describe()` so it returns something like `"T-Shirt [Size M] ($19.99)"`.

In [ ]:
class Apparel(Product):
    # TODO
    pass


# Test
tshirt = Apparel("A001", "T-Shirt", 19.99, "M")
print(tshirt.describe())   # should print: "T-Shirt [Size M] ($19.99)"
print("Is it a Product?", isinstance(tshirt, Product))   # True
print("Is it a Book?   ", isinstance(tshirt, Book))      # False

## Find the bug (L3b): the missing `super().__init__()`

Below is a Book class with one critical line removed. Run the cell and read the error carefully.

In [ ]:
class BookBuggy(Product):
    def __init__(self, id, name, price, author):
        # NOTE: super().__init__(...) is missing
        self.author = author


try:
    buggy_book = BookBuggy("B999", "Some Book", 19.99, "Some Author")
    print("Book created.")
    print("Author:", buggy_book.author)
    print("Price: ", buggy_book.price)   # this is where it goes wrong
except AttributeError as e:
    print("AttributeError:", e)

**Your task:** In one or two sentences, explain *why* removing `super().__init__(id, name, price)` causes `buggy_book.price` to fail. Be specific — what does `super().__init__()` actually do, and what doesn't happen when you leave it out?

**Your explanation:**

*(replace this text)*

## Comprehension check

S5 had this `Order` class:

```python
class Order:
    def __init__(self, customer, cart):
        self.customer = customer
        self.cart = cart
```

S6 has this `Book` class:

```python
class Book(Product):
    def __init__(self, id, name, price, author):
        super().__init__(id, name, price)
        self.author = author
```

Both have multiple objects/values stored on the new instance. **What is the fundamental difference between the relationship `Order` has with `Customer` and the relationship `Book` has with `Product`?**

Use the phrases "has-a" and "is-a" in your answer.

**Your answer:**

*(replace this text)*

## At-home — Claude allowed (L4): a two-level inheritance chain

Build:

- `DigitalProduct(Product)` — adds `download_url` and `file_size_mb`. Override `describe()` to include the file size.
- `Subscription(DigitalProduct)` — adds `renewal_months`. Override `describe()` to mention the renewal period.

You'll need `super().__init__(...)` in each class. Test that `isinstance(subscription, Product)` returns True (it should — inheritance is transitive).

In [ ]:
# TODO: DigitalProduct and Subscription


# Test
sub = Subscription("S001", "Cloud Storage", 9.99, "https://...", 0.0, renewal_months=12)
print(sub.describe())
print("Is a Product?       ", isinstance(sub, Product))         # True
print("Is a DigitalProduct?", isinstance(sub, DigitalProduct))  # True
print("Is a Subscription?  ", isinstance(sub, Subscription))    # True
print("Is a Book?          ", isinstance(sub, Book))            # False

## At-home — marked pass/fail on engagement (L5): when should an inheritance chain stop?

You now have a chain three levels deep: `Product → DigitalProduct → Subscription`. Real codebases sometimes have chains five, six, seven levels deep. **Is that a good idea?**

Imagine you're asked to add `AnnualSubscription(Subscription)`, then `EnterpriseAnnualSubscription(AnnualSubscription)`. Where (if anywhere) does the inheritance chain stop being useful and start being a maintenance problem?

Pick a side and defend it in 4–5 sentences:

- **Side A:** "Keep extending. Every new specialisation deserves a new subclass."
- **Side B:** "This should have been composition from the start. The deeper the chain, the more brittle the design."

This is graded on engagement. Specific examples, concrete arguments, and concrete trade-offs are what we want.

**Your argument:**

*(replace this text)*

---

# Part 2: Magic methods — making your classes feel Pythonic

Magic methods (also called "dunder methods" because of the **d**ouble **under**scores) are special methods Python uses behind the scenes. You don't normally call them directly — they get called when you do something natural like `print(x)` or `x == y`.

## Instructor demo — read

First, see how unhelpful Python is by default:

In [ ]:
class ProductBasic:
    def __init__(self, name, price):
        self.name = name
        self.price = price


p1 = ProductBasic("Notebook", 12.50)
print(p1)              # not very useful
print(p1 == ProductBasic("Notebook", 12.50))   # also not what you'd expect

`print(p1)` shows a memory address — useless. And `p1 == ProductBasic("Notebook", 12.50)` returns `False` even though both have the same name and price. **Python compares object *identity* by default, not values.**

Let's fix both:

In [ ]:
class ProductNice:
    def __init__(self, name, price):
        self.name = name
        self.price = price

    def __str__(self):
        # Used by print() — human-friendly
        return f"{self.name} (${self.price})"

    def __repr__(self):
        # Used in the REPL / debugger — should be more technical, ideally show how to recreate
        return f"ProductNice(name={self.name!r}, price={self.price!r})"

    def __eq__(self, other):
        # Used by ==
        if not isinstance(other, ProductNice):
            return False
        return self.name == other.name and self.price == other.price


p1 = ProductNice("Notebook", 12.50)
p2 = ProductNice("Notebook", 12.50)

print(p1)            # uses __str__
p1                   # uses __repr__ — try evaluating this in a notebook cell on its own
print(p1 == p2)      # uses __eq__

In [ ]:
# Just to see __repr__ in action — evaluating an object on the last line of a cell
p1

## Predict-then-run (L2)

Predict the output of each `print` below **before** running the cell. Pay attention to whether each class defines `__eq__` or not.

**Your predictions:**

- `print(ProductBasic("X", 1) == ProductBasic("X", 1))` → ?
- `print(ProductNice("X", 1) == ProductNice("X", 1))` → ?
- `print(ProductNice("X", 1) == "X")` → ?

*(replace this text with your predictions)*

In [ ]:
print("Basic ==  Basic:", ProductBasic("X", 1) == ProductBasic("X", 1))
print("Nice  ==  Nice: ", ProductNice("X", 1)  == ProductNice("X", 1))
print("Nice  == 'X':   ", ProductNice("X", 1)  == "X")

The first one is the surprise. Even though the two ProductBasic objects have identical attributes, `==` returns False — because without `__eq__`, Python falls back to comparing memory addresses (which are different for each object).

## Your turn (L3): add `__str__` and `__eq__` to Customer

Take your S5 `Customer` class and add `__str__` and `__eq__`:

- `__str__` should return something like `"Alice Chen (gold)"`.
- `__eq__` should compare two Customers as equal if they have the same `email` (assume email is the unique identifier — same email = same person).

In [ ]:
class Customer:
    def __init__(self, name, email, tier):
        self.name = name
        self.email = email
        self.tier = tier

    def apply_discount(self, total):
        if self.tier == "gold":   return total * 0.85
        if self.tier == "silver": return total * 0.90
        return total

    # TODO: __str__
    # TODO: __eq__


# Test
alice1 = Customer("Alice Chen",   "alice@example.com", "gold")
alice2 = Customer("Alice C.",     "alice@example.com", "silver")   # same email, different name/tier
bob    = Customer("Bob Singh",    "bob@example.com",   "gold")

print(alice1)              # should print "Alice Chen (gold)"
print(alice1 == alice2)    # True — same email
print(alice1 == bob)       # False — different email

## Your turn (L3): add `__len__` and `__bool__` to Cart

`__len__` lets your object work with the built-in `len()` function. `__bool__` lets it work with `if x:`.

Add both to a Cart class so that:

- `len(cart)` returns the total quantity across all items (same as the `count_items` method from S5).
- `bool(cart)` returns `True` if there's at least one item, `False` if the cart is empty. This lets you write `if cart: ...` naturally.

In [ ]:
class Cart:
    def __init__(self, items=None):
        self.items = items if items is not None else []

    # TODO: __len__
    # TODO: __bool__


# Test
empty = Cart()
loaded = Cart([{"product_id": "P001", "qty": 2}, {"product_id": "P002", "qty": 1}])

print("len(empty): ", len(empty))     # 0
print("len(loaded):", len(loaded))    # 3

if empty:
    print("empty is truthy")
else:
    print("empty is falsy")            # expected

if loaded:
    print("loaded is truthy")          # expected
else:
    print("loaded is falsy")

## Comprehension check

Without `__eq__`, Python's default behaviour for `x == y` compares something — but it's not the values of the attributes. **What does Python compare by default?**

(Hint: think back to what `print(p1)` showed for the basic class. The answer to "what does `==` compare by default" is closely related to "what does `print` show by default.")

**Your answer:**

*(replace this text)*

## At-home — Claude allowed (L4): a `Money` class

Build a `Money` class that represents a monetary amount in a specific currency. It needs:

- Attributes: `amount` (float) and `currency` (string like `"AUD"`, `"USD"`).
- `__str__` — returns something like `"$12.50 AUD"`.
- `__eq__` — two Money objects are equal if they have the same amount **and** the same currency.
- `__add__` — `Money(10, "AUD") + Money(5, "AUD")` returns `Money(15, "AUD")`. Should raise `ValueError` if the currencies differ.
- `__sub__` — same rules as `__add__`.

In [ ]:
# TODO: Money class


# Test
a = Money(10.00, "AUD")
b = Money(5.00, "AUD")
c = Money(5.00, "USD")

print(a + b)        # $15.00 AUD
print(a == Money(10.00, "AUD"))   # True
print(a == c)                     # False — different currency

try:
    print(a + c)
except ValueError as e:
    print("Caught:", e)

## At-home — marked pass/fail on engagement (L5): cross-currency comparison

You'd like to sort a list of Money objects: `sorted([Money(50, "AUD"), Money(20, "USD"), Money(100, "EUR")])`. To do that, you'd need to implement `__lt__` (less-than). **But:**

- Should `Money(50, "AUD") < Money(100, "USD")` even make sense? At today's exchange rate, maybe AUD 50 is worth more or less than USD 100 — but those exchange rates change every minute.

You have several options:

- **A.** Raise an error on `<` if currencies differ. Cross-currency comparison is meaningless without an exchange rate; refuse to guess.
- **B.** Compare by amount only, ignoring currency. Fast, simple, sometimes wrong.
- **C.** Require the user to pass an exchange-rate lookup into Money. Realistic, but more complex.

**Pick one. Implement it. Justify your choice in 4–5 sentences.** Cover what your design gets right and what it sacrifices.

(For business analytics work, this kind of decision comes up constantly. The "correct" answer depends on what your code is for — there's no universal right answer.)

In [ ]:
# TODO: Money with your chosen __lt__ design


# Test
prices = [Money(50, "AUD"), Money(20, "AUD"), Money(35, "AUD")]
print(sorted(prices))


**Your justification:**

*(replace this text)*

---

# Part 3: `@dataclass` — less boilerplate

Look at your S5 Customer class. Look at the `__init__`. It's almost entirely repetitive: take a parameter, store it as an attribute, take a parameter, store it as an attribute. Then earlier you added `__repr__` and `__eq__` — and those are also mostly boilerplate.

Python has a shortcut. The `@dataclass` decorator writes `__init__`, `__repr__`, and `__eq__` for you, based on type-annotated class attributes.

## Instructor demo — read

In [ ]:
from dataclasses import dataclass


@dataclass
class Customer:
    name: str
    email: str
    tier: str = "bronze"     # default value — bronze unless specified

    def apply_discount(self, total):
        if self.tier == "gold":   return total * 0.85
        if self.tier == "silver": return total * 0.90
        return total


# That's the whole class. Now use it like any other class:
alice = Customer("Alice Chen", "alice@example.com", "gold")
bob   = Customer("Bob Singh",  "bob@example.com")    # tier defaults to bronze

print(alice)                                  # uses auto-generated __repr__
print(alice == Customer("Alice Chen", "alice@example.com", "gold"))   # auto __eq__ — True
print(alice.apply_discount(100))

Compare the line count to your S5 Customer class. The `@dataclass` version is shorter, more readable, and **does more** (gives you a nice `__repr__` and a working `__eq__` for free).

The price: every attribute needs a **type annotation** (the `: str` after `name`). The annotation is just documentation — Python doesn't enforce it at runtime — but `@dataclass` needs it to know what to generate.

## Predict-then-run (L2)

What methods does `@dataclass` auto-generate? Predict, then run.

**Your prediction:**

Which of these methods exist on the new `Customer` class — `__init__`, `__repr__`, `__eq__`, `__str__`, `__lt__`, `__hash__`?

*(replace this text with your predictions)*

In [ ]:
# Inspect: which methods did @dataclass actually add to Customer?
# (A method "added" by @dataclass appears in Customer.__dict__.
#  A method only inherited from object — Python's default — does not.)

methods_to_check = ["__init__", "__repr__", "__eq__", "__str__", "__lt__", "__hash__"]
for m in methods_to_check:
    if m in Customer.__dict__:
        value = Customer.__dict__[m]
        if value is None:
            print(f"{m:<12}: set to None by @dataclass (Customer is now unhashable)")
        else:
            print(f"{m:<12}: ADDED by @dataclass")
    else:
        print(f"{m:<12}: not added — Python falls back to object's default")

Notes on the result:

- `__init__`, `__repr__`, `__eq__` are written for you.
- `__str__` is NOT auto-generated — Python falls back to `__repr__` if you call `str()`. (That's why `print(alice)` worked above.)
- `__lt__` is NOT auto-generated by default. You can opt in with `@dataclass(order=True)`.
- `__hash__` behaviour is subtle and depends on whether the dataclass is frozen.

## Your turn (L3a): rewrite Product as `@dataclass`

Take the S6 `Product` class from Part 1 (`id`, `name`, `price`) and rewrite it as a dataclass.

In [ ]:
from dataclasses import dataclass

@dataclass
class Product:
    # TODO: three type-annotated attributes
    pass


# Test
p = Product("P001", "Notebook", 12.50)
print(p)
print(p == Product("P001", "Notebook", 12.50))

## Your turn (L3b): predict the three frozen-dataclass operations

`@dataclass(frozen=True)` makes the class **immutable** — you can't reassign attributes after construction.

But "immutable" turns out to be a slippery word. Predict what happens for each of the three operations below, then run the cell.

**Your predictions:**

Given `audit = AuditLog(order_id="O001", items=[{"product_id": "P001", "qty": 2}])`, what happens for each operation?

- `audit.order_id = "O002"` (reassign attribute) → ?
- `audit.items.append({"product_id": "P002", "qty": 1})` (mutate internal list) → ?
- `audit.items = []` (reassign attribute to a new value) → ?

*(replace this text with your predictions: for each one, will it succeed, raise an error, or do something else?)*

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class AuditLog:
    order_id: str
    items: list


audit = AuditLog(order_id="O001", items=[{"product_id": "P001", "qty": 2}])

# Operation 1: reassign attribute
print("Operation 1: reassign attribute")
try:
    audit.order_id = "O002"
    print("  Succeeded — audit.order_id is now", audit.order_id)
except Exception as e:
    print(f"  Failed: {type(e).__name__}: {e}")

# Operation 2: mutate internal list (append)
print("\nOperation 2: append to internal list")
try:
    audit.items.append({"product_id": "P002", "qty": 1})
    print("  Succeeded — audit.items is now", audit.items)
except Exception as e:
    print(f"  Failed: {type(e).__name__}: {e}")

# Operation 3: reassign attribute to a new value
print("\nOperation 3: reassign attribute to new value")
try:
    audit.items = []
    print("  Succeeded — audit.items is now", audit.items)
except Exception as e:
    print(f"  Failed: {type(e).__name__}: {e}")

## Comprehension check

**`@dataclass(frozen=True)` does NOT make a class fully immutable.** Operation 2 above succeeded — appending to the internal list worked. In one or two sentences, explain *why*. What is `frozen=True` actually freezing, and what isn't it freezing?

(This is a real-world gotcha. People who think `frozen=True` means "deeply immutable" get bitten in production.)

**Your answer:**

*(replace this text)*

## A note on multiple inheritance

Python lets you write `class C(A, B):` — a class that inherits from two parents. This is **multiple inheritance**.

We're not going to do exercises on it, because:

- The rules for "which parent's method gets used" (the Method Resolution Order, or MRO) get subtle fast.
- Real business-analytics code almost never needs it.
- When it's used badly, multiple inheritance produces some of the most painful bugs in any codebase.

If you ever find yourself wanting to use multiple inheritance, **stop and ask whether composition would work instead.** It almost always does, and the result is usually clearer.

## At-home — Claude allowed (L4): rewrite your S5 mini-project as dataclasses

Take your `Customer`, `Cart`, and `Order` classes from the S5 mini-project. Convert all three to dataclasses. Confirm the behaviour is **identical** — same inputs, same outputs.

Things to think about:

- `Cart.items` and `Customer` references need careful handling — see the docs on `field(default_factory=list)`. (Hint: never use a mutable default like `items: list = []` directly in a dataclass — it has a famous Python gotcha. Find out why.)
- Your methods (`apply_discount`, `compute_subtotal`, `process`) stay the same.

In [ ]:
from dataclasses import dataclass, field

# TODO: rewrite Customer, Cart, Order as dataclasses


# Test — should produce the same summary dict as your S5 mini-project
catalogue = {
    "P001": {"name": "Notebook", "price": 12.50},
    "P002": {"name": "Pen pack", "price": 4.20},
}

alice = Customer("Alice Chen", "alice@example.com", "gold")
alice_cart = Cart(items=[{"product_id": "P001", "qty": 2}])
order = Order(alice, alice_cart)
print(order.process(catalogue))

## At-home — marked pass/fail on engagement (L5): when to use `@dataclass`, and when not

`@dataclass` is great for some classes and overkill for others. In two short paragraphs:

1. **Describe a kind of class where `@dataclass` is the right tool.** Be specific — what makes it a good fit?
2. **Describe a kind of class where `@dataclass` would be the wrong tool.** Same — what makes it a bad fit?

(Hint: dataclasses shine for classes that are mostly "data holders" with little behaviour. They shine less when the class has lots of behaviour, complex initialisation, or inheritance hierarchies with method overrides at multiple levels.)

**Your answer:**

*(replace this text)*

---

# Part 4: Pair programming — a logistics Fleet

You're now going to combine everything from S5 and S6 — inheritance, composition, magic methods, and (optionally) dataclasses — in one exercise.

## The scenario

A logistics company runs a fleet of vehicles. There are three vehicle types — **Trucks**, **Ships**, and **Planes** — each with a different weight capacity. The company wants:

1. A common parent `Vehicle` class with a `capacity()` method (each subclass returns its own capacity).
2. `Truck`, `Ship`, and `Plane` classes that inherit from `Vehicle`.
3. A `Fleet` class that **composes** (has-a list of) Vehicles.
4. A `__str__` method on Fleet that summarises the fleet's total capacity.

## Pair programming protocol

If you're in class, work in pairs:

1. **Driver** writes; **Observer** reviews. Switch every 5 minutes.
2. The Observer's job is not to take over — it's to ask good questions. "Why did you choose `super().__init__` here?" "What happens if we add a non-Vehicle to the fleet?"
3. Test as you go. Run after every class.

## L4: build the Fleet (in-class, Claude allowed if you're stuck)

Build the four classes below. Vehicle should be a normal class (not a dataclass — we want a method, not data); Fleet can be either, your choice.

In [ ]:
# TODO: Vehicle (base class) with a capacity() method that subclasses override


# TODO: Truck, Ship, Plane — each subclass with its own capacity()


# TODO: Fleet — composes a list of vehicles; has __str__


# Test
truck = Truck(weight_limit=20000)
ship  = Ship(weight_limit=50000)
plane = Plane(weight_limit=10000)

fleet = Fleet()
fleet.add_vehicle(truck)
fleet.add_vehicle(ship)
fleet.add_vehicle(plane)

print(fleet)            # something like "Fleet of 3 vehicles — total capacity 80000 kg"
print(len(fleet))       # 3 — if you decided to add __len__ as well (optional)

## Stretch (L4, optional): refactor Fleet as a `@dataclass`

Can you make Fleet a dataclass? The `vehicles` attribute is a list, so you'll need `field(default_factory=list)` instead of a plain default. Keep `add_vehicle` and `__str__` as methods.

In [ ]:
# TODO (stretch): @dataclass version of Fleet


# Test as before


## At-home — marked pass/fail on engagement (L5): the MaintenanceSchedule design decision

The company wants to track maintenance for each vehicle: a list of `MaintenanceRecord`s (date + description + cost), plus a method to compute total maintenance cost.

You have **three reasonable design options**:

- **Option A — composition.** Each `Vehicle` HAS-A `MaintenanceSchedule`. The schedule is a separate class that holds the records. `vehicle.schedule.total_cost()`.
- **Option B — inheritance.** A `MaintainedVehicle(Vehicle)` subclass adds maintenance tracking. Only some vehicles need maintenance tracking; they use the subclass.
- **Option C — both.** A `MaintenanceSchedule` class exists separately, but `Vehicle` itself holds a reference to one. Allows the schedule to be shared, replaced, or compared independently.

**Pick one. Implement it. Justify your choice in 5–6 sentences.** Address:

- What does your design make easy?
- What does your design make harder (every design has trade-offs)?
- If the company later decides "actually, *every* vehicle needs maintenance tracking," does your design still hold up, or does it need refactoring?

This is the kind of decision real software designers face every day. There is no correct answer. We are grading whether you engage with the trade-offs.

In [ ]:
# TODO: implement your chosen design


**Your justification:**

*(replace this text)*

---

## Final reflection (required) — end of the OOP block

This is the last reflection in the Week 3 OOP block. Three short answers:

1. **Across S5 and S6, what was the single hardest concept?** Was it `self`? `super().__init__()`? Magic methods? Composition vs inheritance? Frozen dataclasses being not-fully-frozen? Be specific.

2. **Pick one method or class you wrote in S5 or S6 and explain it to a hypothetical teammate who only knows S1–S4.** Paste the code, walk through what each line does, and explain *why* it's structured this way. 5–6 sentences.

3. **If you used Claude or another AI tool across S5 and S6:** Paste one or two of your most useful prompts. Then a short paragraph (3–4 sentences): what patterns did you notice in how you used the AI? When did it help most? When did its answers mislead you or use features you hadn't learned?

**Your final reflection:**

*(replace this text)*

---

## End of the OOP block — what's next

You've now finished the four-session OOP block. By this point you can:

- Build classes with `class`, `__init__`, `self`, attributes, and methods (S5).
- Connect classes via composition — has-a relationships (S5).
- Use `isinstance()` carefully (S5).
- Specialise classes via inheritance — is-a relationships (S6).
- Make classes Pythonic with magic methods (S6).
- Reduce boilerplate with `@dataclass` and reason about the limits of `frozen=True` (S6).

**Where this goes next.** The OOP block is the foundation for the rest of the unit:

- **Pandas / DataFrames.** A `DataFrame` is a class. It uses magic methods extensively — `__getitem__` for `df[col]`, `__len__` for `len(df)`, `__repr__` for the pretty-printed table you see in notebooks. Now that you've written `__getitem__`'s cousins yourself, DataFrames will feel less magical.
- **File I/O at scale.** `csv.DictReader` and `json.load()` produce dicts that map naturally onto class instances. You'll see this pattern repeatedly.
- **Designing your own code.** The composition-vs-inheritance, mutable-vs-immutable, eager-vs-lazy decisions you made in this notebook are the same decisions software designers make every day. The vocabulary you have now is the vocabulary the field uses.

Good work.